# Using an LLM API to improve Task 1 (sentiment analysis)

**What this notebook does.** It adds a large language model (NVIDIA's hosted Nemotron, reached through an API) to the sentiment work, in three ways:

1. **A new comparison point:** a zero-shot and a few-shot LLM classifier, scored on the same test reviews as TF-IDF and RoBERTa.
2. **Label-noise check:** where the LLM disagrees with the star rating, we read the reviews. This shows why neutral is so hard.
3. **A hybrid:** use RoBERTa when it is confident, and ask the LLM only when it is not. We simulate this at no extra cost.

**What it does not do.** It does not replace the fine-tuned model. The Task 1 labels come from star ratings (1-2 negative, 3 neutral, 4-5 positive), so RoBERTa learned how reviewers *use stars*, while an LLM judges the *text*. They can disagree exactly where stars and text disagree. Whether the LLM scores higher against star-based labels is an open question that this notebook measures.

**Fair comparison.** Calling an API for all 5,190 test reviews would be slow, so we score a stratified subset (all rare-class reviews, a sample of positives). Every model is scored on the *same* subset, and scores are weighted to undo the over-sampling, so they reflect the real class mix.

**Cost and privacy.** Each review is one API call. Review texts are sent to NVIDIA's service. Start with `QUICK = True`.

In [ ]:
# %pip install -q openai scikit-learn pandas matplotlib

In [1]:
import os, re, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor
from IPython.display import display
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

SEED = 42
DATA_DIR = "."                       # folder with train.csv and test.csv (made by the baseline notebook)
LABELS = ["negative", "neutral", "positive"]

API_BASE_URL = "https://integrate.api.nvidia.com/v1"
API_MODEL = "nvidia/nemotron-3-ultra-550b-a55b"
MAX_WORKERS = 4                      # calls in parallel; lower it if you hit rate limits

QUICK = True                         # True: about 250 reviews (500 calls). False: the full subset, about 800 reviews (1,600 calls)
N_PER_CLASS = {"negative": 60, "neutral": 60, "positive": 130} if QUICK else {"negative": 122, "neutral": 225, "positive": 450}

ROBERTA_ID = ""                      # your fine-tuned model on the Hub, e.g. "Roberto-Vargas/<repo>". Leave empty to skip RoBERTa.

## 1. The test subset

We take reviews from `test.csv` (never used for training) and keep class sizes from `N_PER_CLASS`. Because negative and neutral reviews are over-represented in the subset, each review gets a **weight** equal to how many real test reviews it stands for. Weighted scores then match what the full test set would show.

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

parts = []
for lab, n in N_PER_CLASS.items():
    pool = test[test["sentiment"] == lab]
    parts.append(pool.sample(min(n, len(pool)), random_state=SEED))
sub = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

test_counts, sub_counts = test["sentiment"].value_counts(), sub["sentiment"].value_counts()
sub["weight"] = sub["sentiment"].map(lambda s: test_counts[s] / sub_counts[s])
print("Subset size:", len(sub), "of", len(test), "test reviews")
print(sub["sentiment"].value_counts().to_dict())

FileNotFoundError: [Errno 2] No such file or directory: './train.csv'

## 2. The LLM classifier

The model gets one review and must answer with a single word. We try two prompts:

- **Zero-shot:** only the definitions of the three classes.
- **Few-shot:** the same, plus six example reviews taken from the **training** set (two per class), so there is no leakage from the test set.

Temperature is 0 so the answers are repeatable. Answers are saved to a file as they arrive, so if the run stops you can run the cell again and it continues where it stopped.

The key is requested with a hidden prompt. Do not paste it into a code cell or into a chat.

In [ ]:
from getpass import getpass
from openai import OpenAI

client = OpenAI(base_url=API_BASE_URL, api_key=os.environ.get("NVIDIA_API_KEY") or getpass("Paste your NVIDIA API key (hidden): "))

In [ ]:
SYSTEM = ("You classify the sentiment of Amazon product reviews. Answer with exactly one word: negative, neutral or positive.\n"
          "negative = the reviewer is mostly unhappy (typically 1-2 stars). "
          "neutral = mixed or lukewarm, with both good and bad points (typically 3 stars). "
          "positive = the reviewer is mostly happy (typically 4-5 stars). Judge the review text only.")

def pick_examples(k=2):
    ex = []
    for lab in LABELS:
        pool = train[(train["sentiment"] == lab) & train["text"].str.len().between(60, 250)]
        ex += [(t, lab) for t in pool.sample(k, random_state=SEED)["text"]]
    random.Random(SEED).shuffle(ex)             # mixed order, so the last example does not bias the answer
    return ex

EXAMPLES = pick_examples()

def build_messages(text, few_shot):
    msgs = [{"role": "system", "content": SYSTEM}]
    if few_shot:
        for t, lab in EXAMPLES:
            msgs += [{"role": "user", "content": f"Review: {t}"}, {"role": "assistant", "content": lab}]
    msgs.append({"role": "user", "content": f"Review: {text}"})
    return msgs

def parse(answer):
    m = re.search(r"negative|neutral|positive", answer.lower())
    return m.group(0) if m else None

def ask(text, few_shot, tries=4):
    for attempt in range(tries):
        try:
            r = client.chat.completions.create(
                model=API_MODEL, messages=build_messages(text, few_shot), temperature=0, max_tokens=16,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}})
            return parse(r.choices[0].message.content or "")
        except Exception as err:                       # rate limits and network hiccups: wait and retry
            if attempt == tries - 1:
                print(f"gave up on one review ({type(err).__name__})")
                return None
            time.sleep(5 * 2 ** attempt)

def classify_all(texts, few_shot, cache_path):
    cache = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
    todo = [t for t in dict.fromkeys(texts) if t not in cache]
    print(f"{len(todo)} calls to make ({len(texts) - len(todo)} already saved)")
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        for t, lab in zip(todo, pool.map(lambda t: ask(t, few_shot), todo)):
            if lab:
                cache[t] = lab
    json.dump(cache, open(cache_path, "w"))
    return [cache.get(t) for t in texts]

In [ ]:
tag = API_MODEL.split("/")[-1]
sub["zero_shot"] = classify_all(sub["text"].tolist(), False, f"llm_zero_shot_{tag}.json")
sub["few_shot"] = classify_all(sub["text"].tolist(), True, f"llm_few_shot_{tag}.json")
print("Unparsed answers:", sub[["zero_shot", "few_shot"]].isna().sum().to_dict())

If many answers are unparsed, the model may be spending its token budget on reasoning text. Raise `max_tokens` in `ask` (for example to 64) and run the cell again; saved answers are kept.

## 3. The other models, on the same subset

**TF-IDF + logistic regression** is refitted here in seconds, with the settings that won on validation in the baseline notebook. **RoBERTa** is loaded from the Hub if you set `ROBERTA_ID`. We keep its confidence too, which the hybrid in section 6 needs.

In [ ]:
tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, strip_accents="unicode")),
    ("clf", LogisticRegression(C=10, class_weight="balanced", max_iter=1000, random_state=SEED)),
]).fit(train["text"], train["label"])
sub["tfidf"] = [LABELS[i] for i in tfidf.predict(sub["text"])]

In [ ]:
if ROBERTA_ID:
    from transformers import pipeline
    rob = pipeline("text-classification", model=ROBERTA_ID, top_k=None, truncation=True, max_length=128)
    rename = {"LABEL_0": "negative", "LABEL_1": "neutral", "LABEL_2": "positive"}
    preds, confs = [], []
    for out in rob(sub["text"].tolist(), batch_size=16):
        out = out[0] if out and isinstance(out[0], list) else out
        scores = {rename.get(d["label"], d["label"]): d["score"] for d in out}
        best = max(scores, key=scores.get)
        preds.append(best); confs.append(scores[best])
    sub["roberta"], sub["roberta_conf"] = preds, confs
else:
    print("ROBERTA_ID is empty: skipping RoBERTa. Set it in the settings cell to include it.")

## 4. Scores

**Macro-F1** is the headline metric, as in Task 1. Scores are weighted to match the real class mix (`weighted`). The `balanced` column ignores the weights, so each class in the subset counts as sampled, which shows how a model treats the rare classes. An unparsed LLM answer counts as wrong.

In [ ]:
def scores(col):
    y, p = sub["sentiment"], sub[col].fillna("none")
    w = sub["weight"]
    f1 = f1_score(y, p, labels=LABELS, average=None, sample_weight=w)
    return {"macro_f1 (weighted)": f1_score(y, p, labels=LABELS, average="macro", sample_weight=w),
            "macro_f1 (balanced)": f1_score(y, p, labels=LABELS, average="macro"),
            "accuracy (weighted)": accuracy_score(y, p, sample_weight=w),
            "negative_f1": f1[0], "neutral_f1": f1[1], "positive_f1": f1[2]}

models = {"TF-IDF + LR": "tfidf", "LLM zero-shot": "zero_shot", "LLM few-shot": "few_shot"}
if "roberta" in sub:
    models["RoBERTa (fine-tuned)"] = "roberta"
table = pd.DataFrame({name: scores(col) for name, col in models.items()}).T.round(3)
display(table)

In [ ]:
best_col = models[table["macro_f1 (weighted)"].idxmax()]
y, p = sub["sentiment"], sub[best_col].fillna("none")
print("Best model on this subset:", table["macro_f1 (weighted)"].idxmax(), "\n")
report = pd.DataFrame(classification_report(y, p, labels=LABELS, sample_weight=sub["weight"], zero_division=0, output_dict=True)).T
report = report.drop(index="accuracy")                               # accuracy is already in the table above
report["support"] = report["support"].round(0).astype(int)        # weighted counts: roughly the real test-set sizes
display(report.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay.from_predictions(y, p, labels=LABELS, display_labels=LABELS, cmap="Blues", ax=ax[0], colorbar=False)
ax[0].set_title("Counts in the subset")
ConfusionMatrixDisplay.from_predictions(y, p, labels=LABELS, display_labels=LABELS, cmap="Blues", normalize="true", values_format=".0%", ax=ax[1], colorbar=False)
ax[1].set_title("Share of each true class")
plt.tight_layout(); plt.show()

## 5. Where the LLM and the stars disagree

This is the most useful part for the write-up. If the LLM calls a 2-star review positive, either the model is wrong or the reviewer's stars do not match their words. Read the examples and decide.

The table shows, for each star rating, what share of reviews the LLM put in each class. The 3-star row shows how split the "neutral" class really is.

In [ ]:
llm = "few_shot"
agree = pd.crosstab(sub["rating"], sub[llm], normalize="index").reindex(columns=LABELS).round(2)
agree.insert(0, "reviews", sub["rating"].value_counts().sort_index())
display(agree)

In [ ]:
def show(true, said, n=3):
    rows = sub[(sub["sentiment"] == true) & (sub[llm] == said)]
    print(f"\n=== stars say {true}, LLM says {said}: {len(rows)} reviews ===")
    for _, r in rows.sample(min(n, len(rows)), random_state=SEED).iterrows():
        print(f"[{r['rating']} stars] {r['text'][:260]}\n")

show("negative", "positive")
show("positive", "negative")
show("neutral", "positive")
show("neutral", "negative")

## 6. A hybrid: RoBERTa first, the LLM only when RoBERTa is unsure

For each confidence threshold, reviews where RoBERTa is **less** confident than the threshold are sent to the LLM, and the rest keep RoBERTa's answer. Because we already have the LLM's answer for every review in the subset, we can simulate every threshold without more calls.

- `share_sent_to_llm` is how many calls the hybrid would need, as a share of all reviews.
- Threshold 0 is plain RoBERTa. A threshold above 1 sends everything to the LLM.

If the hybrid beats both RoBERTa and the LLM at a modest `share_sent_to_llm`, it is a real improvement. If not, the answer is also worth reporting.

In [ ]:
if "roberta" in sub:
    rows = []
    for tau in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 1.01]:
        use_llm = ((sub["roberta_conf"] < tau) & sub[llm].notna()).to_numpy()
        pred = np.where(use_llm, sub[llm], sub["roberta"])
        rows.append({"threshold": tau,
                     "share_sent_to_llm": round(float(np.average(use_llm, weights=sub["weight"])), 3),
                     "macro_f1 (weighted)": round(f1_score(sub["sentiment"], pred, labels=LABELS, average="macro", sample_weight=sub["weight"]), 3),
                     "neutral_f1": round(f1_score(sub["sentiment"], pred, labels=LABELS, average=None, sample_weight=sub["weight"])[1], 3)})
    hybrid = pd.DataFrame(rows).set_index("threshold")
    display(hybrid)
    hybrid["macro_f1 (weighted)"].plot(marker="o", figsize=(6, 3.2), title="Hybrid macro-F1 by confidence threshold")
    plt.ylabel("macro-F1"); plt.show()
else:
    print("Needs ROBERTA_ID (section 3).")

In [ ]:
sub.drop(columns=["weight"]).to_csv("llm_sentiment_results.csv", index=False)
print("Saved llm_sentiment_results.csv")

## How to read and report the results

- **The subset is small.** With only a few dozen negative and neutral reviews, differences of a few F1 points can be chance. Run with `QUICK = False` before drawing conclusions, and describe small gaps as "similar".
- **Stars are a noisy target.** If the LLM scores below RoBERTa, that does not mean it reads worse: RoBERTa was trained to imitate the star-based labels, and the LLM was not. The disagreement examples in section 5 show which side is closer to the text.
- **Practical limits.** The LLM costs a call per review and sends text to a third party. RoBERTa runs on your own hardware in a second. That is why the hybrid matters: it keeps most predictions local.
- **Keep the deployed model as it is** unless the hybrid clearly wins. If you do add the LLM to the demo app, it would need the API key stored as a Space secret, never in the code.